- install

In [3]:
!pip install pillow pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 43.3 MB/s eta 0:00:00


- mount 및 경로

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import re
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from collections import defaultdict

register_heif_opener()

folder들 경로만 맞게 수정하시면 됩니다. input_folder의 경우 공유 드라이브 바로가기 생성해서 사용하시면 됩니다

In [8]:
input_folder = "/content/drive/MyDrive/Colab Notebooks/오픈소스프로그래밍 데이터셋"
output_folder = "/content/processed_images"

target_size = 640
jpg_quality = 95

- jpg로 변환 후 클래스 갯수 반환

In [11]:
valid_extensions = [".jpg", ".jpeg", ".png", ".heic", ".webp", ".bmp"]

os.makedirs(output_folder, exist_ok=True)

image_paths = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        ext = os.path.splitext(file.lower())[1]
        if ext in valid_extensions:
            image_paths.append(os.path.join(root, file))

image_paths.sort()
print(f"총 {len(image_paths)}개의 이미지 파일을 찾았습니다.")

class_counts = defaultdict(int)

for input_path in image_paths:
    try:
        img = Image.open(input_path)
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img.thumbnail((target_size, target_size))

        new_img = Image.new("RGB", (target_size, target_size), (255, 255, 255))
        x = (target_size - img.width) // 2
        y = (target_size - img.height) // 2
        new_img.paste(img, (x, y))

        # 원본 파일명에서 클래스명 추출 (숫자_확장자 제거)
        filename = os.path.basename(input_path)
        name_without_ext = os.path.splitext(filename)[0]

        # 클래스 목록
        valid_classes = {"fist", "gun", "heart", "number3", "ok", "one", "palm", "rock", "thumb", "promise",
                          "number3_modified", "three"}

        name_no_number = re.sub(r'_\d+$', '', name_without_ext)

        # number3_modified 먼저 처리
        remainder = name_no_number
        if remainder.startswith("number3_modified"):
            class_counts["number3_modified"] += 1
            remainder = remainder[len("number3_modified"):].lstrip('_')

        for word in remainder.split('_'):
            if word in valid_classes:
                class_counts[word] += 1
            elif word == "thiumb":
                class_counts["thumb"] += 1

        # 파일명은 그대로 유지, 확장자만 jpg로
        new_filename = f"{name_without_ext}.jpg"
        output_path = os.path.join(output_folder, new_filename)

        new_img.save(output_path, "JPEG", quality=jpg_quality)
        print(f"{filename} → {new_filename}")

    except Exception as e:
        print(f"처리 실패: {input_path}")
        print("에러:", e)

print("\n전처리 완료!")
print("저장 위치:", output_folder)

print("\n=== 클래스별 이미지 개수 ===")
for class_name, count in sorted(class_counts.items()):
    print(f"{class_name}: {count}개")
print(f"\n총합: {sum(class_counts.values())}개")

총 453개의 이미지 파일을 찾았습니다.
ok_001.PNG → ok_001.jpg
ok_002.PNG → ok_002.jpg
ok_003.PNG → ok_003.jpg
ok_004.PNG → ok_004.jpg
ok_005.PNG → ok_005.jpg
ok_006.PNG → ok_006.jpg
ok_007.PNG → ok_007.jpg
ok_008.PNG → ok_008.jpg
ok_009.PNG → ok_009.jpg
ok_010.PNG → ok_010.jpg
ok_011.png → ok_011.jpg
ok_012.png → ok_012.jpg
ok_030.jpg → ok_030.jpg
ok_031.jpg → ok_031.jpg
ok_032.jpg → ok_032.jpg
ok_033.jpg → ok_033.jpg
ok_034.jpg → ok_034.jpg
ok_036.jpg → ok_036.jpg
ok_037.jpg → ok_037.jpg
ok_038.jpg → ok_038.jpg
ok_044.jpg → ok_044.jpg
ok_045.png → ok_045.jpg
ok_046.png → ok_046.jpg
ok_047.jpg → ok_047.jpg
ok_048.jpg → ok_048.jpg
ok_049.jpg → ok_049.jpg
palm_001.PNG → palm_001.jpg
palm_002.PNG → palm_002.jpg
palm_003.PNG → palm_003.jpg
palm_004.PNG → palm_004.jpg
palm_005.PNG → palm_005.jpg
palm_006.PNG → palm_006.jpg
palm_007.PNG → palm_007.jpg
palm_008.PNG → palm_008.jpg
palm_009.PNG → palm_009.jpg
palm_010.PNG → palm_010.jpg
palm_011.PNG → palm_011.jpg
palm_012.PNG → palm_012.jpg
palm_049.PNG → pa

In [ ]:
import shutil
from google.colab import files

zip_name = "/content/processed_images"
shutil.make_archive(zip_name, "zip", output_folder)
files.download("/content/processed_images.zip")